In [16]:
import torch.nn as nn
import torch
from tqdm import tqdm


In [2]:
import numpy as np
import glob as glob

In [4]:
from PIL import Image
TRAIN_BATCH_SIZE = 128
TEST_BATCH_SIZE = 1
from torch.utils.data import DataLoader, Dataset

In [5]:
# The SRCNN dataset module. 
class SRCNNDataset(Dataset):
    def __init__(self, image_paths, label_paths):
        self.all_image_paths = glob.glob(f"{image_paths}/*")
        self.all_label_paths = glob.glob(f"{label_paths}/*") 
    def __len__(self):
        return (len(self.all_image_paths))
    def __getitem__(self, index):
        image = Image.open(self.all_image_paths[index]).convert('RGB')
        label = Image.open(self.all_label_paths[index]).convert('RGB')
        image = np.array(image, dtype=np.float32)
        label = np.array(label, dtype=np.float32)
        image /= 255.
        label /= 255.
        image = image.transpose([2, 0, 1])
        label = label.transpose([2, 0, 1])
        return (
            torch.tensor(image, dtype=torch.float),
            torch.tensor(label, dtype=torch.float)
        )

In [6]:
def give_datasets(
    train_image_paths, train_label_paths,
    valid_image_path, valid_label_paths):
    dataset_train = SRCNNDataset(
        train_image_paths, train_label_paths
    )
    dataset_valid = SRCNNDataset(
        valid_image_path, valid_label_paths
    )
    return dataset_train, dataset_valid
# Prepare the data loaders 
def give_dataloaders(dataset_train, dataset_valid):
    train_loader = DataLoader(
        dataset_train, 
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True
    )
    valid_loader = DataLoader(
        dataset_valid, 
        batch_size=TEST_BATCH_SIZE,
        shuffle=False
    )
    return train_loader, valid_loader


In [7]:
import torch.nn as nn
import torch.nn.functional as F
# implementing the srcnn model as on the paper, with some modifications
# 9-1-5 kernel sizes
class SRCNN(nn.Module):
    def __init__(self):
        super(SRCNN, self).__init__()
        self.conv1 = nn.Conv2d(
            3, 64, kernel_size=9, stride=(1, 1), padding=(2, 2)
        )
        self.conv2 = nn.Conv2d(
            64, 32, kernel_size=1, stride=(1, 1), padding=(2, 2)
        )
        self.conv3 = nn.Conv2d(
            32, 3, kernel_size=5, stride=(1, 1), padding=(2, 2)
        )
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        return x

In [15]:
import torch.optim as optim
import os
EPOCH=250
LR=0.003
#specifying the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Constant paths of training and validation data.
TRAIN_LABEL_PATHS = '../input/91_hr'
TRAIN_IMAGE_PATHS = '../input/91_lr'
VALID_LABEL_PATHS = '../input/test_hr'
VALID_IMAGE_PATHS = '../input/test_bicubic_rgb_2x'
SAVE_VALIDATION_RESULTS = True
os.makedirs('../outputs/valid_results', exist_ok=True)

In [9]:
model = SRCNN().to(device)

In [14]:
optimizer = optim.Adam(model.parameters(), lr=LR)
# Loss function. 
criterion = nn.MSELoss()
# using MSE loss as mentioned in the paper.
dataset_train, dataset_valid = give_datasets(
    TRAIN_IMAGE_PATHS, TRAIN_LABEL_PATHS,
    VALID_IMAGE_PATHS, VALID_LABEL_PATHS
)
train_loader, valid_loader = give_dataloaders(dataset_train, dataset_valid)
print(f"Training samples: {len(dataset_train)}")
print(f"Validation samples: {len(dataset_valid)}")

Training samples: 22227
Validation samples: 19


In [18]:

import math
# defining psnr
def psnr(label, outputs, max_val=1.):
    """
    Compute Peak Signal to Noise Ratio (the higher the better).
    PSNR = 20 * log10(MAXp) - 10 * log10(MSE)."""
    
    label = label.cpu().detach().numpy()
    outputs = outputs.cpu().detach().numpy()
    diff = outputs - label
    rmse = math.sqrt(np.mean((diff) ** 2))
    if rmse == 0:
        return 100
    else:
        PSNR = 20 * math.log10(max_val / rmse)
        return PSNR

In [21]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision.utils import save_image
plt.style.use('ggplot')

def save_plot(train_loss, val_loss, train_psnr, val_psnr):
    # Loss plots.
    '''saving the loss and psnr plots'''
    plt.figure(figsize=(10, 7))
    plt.plot(train_loss, color='orange', label='train loss')
    plt.plot(val_loss, color='red', label='validataion loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('../outputs/loss.png')
    plt.close()
    # PSNR plots.
    plt.figure(figsize=(10, 7))
    plt.plot(train_psnr, color='green', label='train PSNR dB')
    plt.plot(val_psnr, color='blue', label='validataion PSNR dB')
    plt.xlabel('Epochs')
    plt.ylabel('PSNR (dB)')
    plt.legend()
    plt.savefig('../outputs/psnr.png')
    plt.close()

def save_model_state(model):
    # save the model to disk
    '''saving intermediate model state to disc'''
    print('Saving model...')
    torch.save(model.state_dict(), '../outputs/model.pth')
def save_model(epochs, model, optimizer, criterion):
    """
    Function to save the trained model to disk.
    """
    # Remove the last model checkpoint if present.
    torch.save({
                'epoch': epochs+1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': criterion,
                }, f"../outputs/model_ckpt.pth")
    
def save_validation_results(outputs, epoch, batch_iter):
    """
    Function to save the validation reconstructed images.
    """
    save_image(
        outputs, 
        f"../outputs/valid_results/val_sr_{epoch}_{batch_iter}.png"
    )

In [22]:
def train(model, dataloader):
    model.train()
    running_loss = 0.0
    running_psnr = 0.0
    for num, data in tqdm(enumerate(dataloader), total=len(dataloader)):
        image_data = data[0].to(device)
        label = data[1].to(device)
        
        # Zero grad the optimizer.
        optimizer.zero_grad()
        outputs = model(image_data)
        loss = criterion(outputs, label)
        # Backpropagation.
        loss.backward()
        # Update the parameters.
        optimizer.step()
        # Add loss of each item (total items in a batch = batch size).
        running_loss += loss.item()
        # Calculate batch psnr (once every `batch_size` iterations).
        batch_psnr =  psnr(label, outputs)
        running_psnr += batch_psnr
    final_loss = running_loss/len(dataloader.dataset)
    final_psnr = running_psnr/len(dataloader)
    return final_loss, final_psnr
def validate(model, dataloader, epoch):
    model.eval()
    running_loss = 0.0
    running_psnr = 0.0
    with torch.no_grad():
        for bi, data in tqdm(enumerate(dataloader), total=len(dataloader)):
            image_data = data[0].to(device)
            label = data[1].to(device)
            
            outputs = model(image_data)
            loss = criterion(outputs, label)
            # Add loss of each item (total items in a batch = batch size) .
            running_loss += loss.item()
            # Calculate batch psnr (once every `batch_size` iterations).
            batch_psnr = psnr(label, outputs)
            running_psnr += batch_psnr
            # For saving the batch samples for the validation results
            
            if SAVE_VALIDATION_RESULTS and (epoch % 25) == 0:
                save_validation_results(outputs, epoch, bi)
    final_loss = running_loss/len(dataloader.dataset)
    final_psnr = running_psnr/len(dataloader)
    return final_loss, final_psnr

In [23]:
import time 
train_loss, val_loss = [], []
train_psnr, val_psnr = [], []
start = time.time()
for epoch in range(EPOCH):
    print(f"Epoch {epoch + 1} of {EPOCH}")
    train_epoch_loss, train_epoch_psnr = train(model, train_loader)
    val_epoch_loss, val_epoch_psnr = validate(model, valid_loader, epoch+1)
    print(f"Train PSNR: {train_epoch_psnr:.3f}")
    print(f"Val PSNR: {val_epoch_psnr:.3f}")
    train_loss.append(train_epoch_loss)
    train_psnr.append(train_epoch_psnr)
    val_loss.append(val_epoch_loss)
    val_psnr.append(val_epoch_psnr)
    
    # Save model with all information every 100 EPOCH. Can be used 
    # resuming training.
    if (epoch+1) % 25 == 0:
        save_model(epoch, model, optimizer, criterion)
    # Save the model state dictionary only every epoch. Small size, 
    # can be used for inference.
    save_model_state(model)
    # Save the PSNR and loss plots every epoch.
    save_plot(train_loss, val_loss, train_psnr, val_psnr)
end = time.time()
print(f"Finished training in: {((end-start)/60):.3f} minutes") 

Epoch 1 of 250


100%|██████████| 19/19 [00:01<00:00, 11.57it/s]


Train PSNR: 28.412
Val PSNR: 28.064
Saving model...
Epoch 2 of 250


100%|██████████| 19/19 [00:01<00:00, 11.95it/s]


Train PSNR: 28.397
Val PSNR: 27.879
Saving model...
Epoch 3 of 250


...

Train PSNR: 29.223
Val PSNR: 29.361
Saving model...
Epoch 249 of 250


100%|██████████| 19/19 [00:02<00:00,  7.01it/s]


Train PSNR: 29.198
Val PSNR: 29.432
Saving model...
Epoch 250 of 250


100%|██████████| 19/19 [00:03<00:00,  4.83it/s]


Train PSNR: 29.066
Val PSNR: 29.373
Saving model...
Finished training in: 110.357 minutes


In [ ]:
# So the final time taken is 110 minutes.
# The final PSNR is around 29.5